In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib as mpl
from matplotlib.ticker import AutoLocator, AutoMinorLocator, LogLocator
import glob
from scipy.interpolate import griddata
from pathlib import Path
import h5py
import sys
from pathlib import Path
import os

# Where am I running?
try:
    # Normal script
    here = Path(__file__).resolve().parent
except NameError:
    # Notebook / REPL
    here = Path.cwd()

phys_const_path = (here / '..' / 'phys_const').resolve()
sys.path.append(str(phys_const_path))

nsm_plots_path = (here / '..' / 'nsm_plots').resolve()
sys.path.append(str(nsm_plots_path))

nsm_plots_postproc = (here / '..' / 'nsm_instabilities').resolve()
sys.path.append(str(nsm_plots_postproc))

import phys_const as pc
import plot_functions as pf

### Studying the trend of quantum coherence in a global post-merger accretion disk as a function of the attenuation factor. Simulation with 92 PPEBPC and 1 cubic-kilometer cells.

In [ ]:
# # HAR
har_rho_eu_48_48=8.237993097357763e-06
har_rho_eu_32_16=1.6913100570908646e-09
# np.average(rho_eu[5:40,16])=1.8255508397170877e-09

# # HSR
hsr_rho_eu_96_60=6.839131272287458e-06
hsr_rho_eu_64_32=2.3495318539521587e-07
# np.average(rho_eu[5:80,32])=2.4813035321102116e-07

In [ ]:
eta = np.array([1e-5,1e-6,1e-7,1e-8,1e-9,1e-10,1e-11,1e-12])
rho_eu_48_48 = np.array([8.28936886418095e-06, 2.8740007172122207e-07, 2.3136689954208057e-08, 2.81132878168119e-09, 2.8437389366196645e-10, 2.8440521140415826e-11, 2.844055244298335e-12, 2.8440552756007335e-13])

log_eta = np.log10(eta)
log_rho_eu_48_48 = np.log10(rho_eu_48_48)

# Fit a straight line: log_frequency = slope * log_eta + intercept
slope, intercept = np.polyfit(log_eta, log_rho_eu_48_48, 1)
print(f"Slope: {slope}, Intercept: {intercept}")

# Create fit line
eta_fit = np.logspace(1, -12, 100)
rho_eu_48_48_fit = 10**(slope * np.log10(eta_fit) + intercept)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(np.log10(eta_fit), np.log10(rho_eu_48_48_fit), label='Linear fit extrapolated', linewidth=3, zorder=1)
ax.plot(np.log10(eta), np.log10(rho_eu_48_48), label='Emu', marker='o', linewidth=3, markersize=12, zorder=2)
ax.scatter([np.log10(1e-5)], [np.log10(har_rho_eu_48_48)], label='EMU HAR', marker='x', s=200, c='black', zorder=3)
ax.scatter([np.log10(1e-5)], [np.log10(hsr_rho_eu_96_60)], label=r'EMU HSR + $\eta_\mathrm{matter}=1e-2$', marker='x', s=200, c='red', zorder=3)
ax.axhline(0, color='gray', linewidth=3, linestyle='--')
ax.axvline(0, color='gray', linewidth=3, linestyle='--')
ax.set_xlabel(r'$\log \eta$')
ax.set_ylabel(r'$\log \rho_{eu}$')
ax.set_title('Quantum coherence in the polar region')
leg = ax.legend(framealpha=0.0, ncol=1, fontsize=20)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.show()
plt.close(fig)

In [ ]:
eta = np.array([1e-5,1e-6,1e-7, 1e-8,1e-9,1e-10,1e-11,1e-12])
rho_eu_32_16 = np.array([2.0949725700233037e-09, 1.694172875250579e-09, 1.6314031395045477e-09, 5.590936886097058e-10, 6.189274147216006e-11, 6.195684651552251e-12, 6.195748792864933e-13, 6.195749434281654e-14])

log_eta = np.log10(eta)
log_rho_eu_32_16 = np.log10(rho_eu_32_16)

# Fit a straight line: log_rho_eu_32_16 = slope * log_eta + intercept
slope, intercept = np.polyfit(log_eta, log_rho_eu_32_16, 1)
print(f"Slope: {slope}, Intercept: {intercept}")

# Create fit line
eta_fit = np.logspace(1, -12, 100)
rho_eu_fit_32_16 = 10**(slope * np.log10(eta_fit) + intercept)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(np.log10(eta_fit), np.log10(rho_eu_fit_32_16), label='Linear fit extrapolated', linewidth=3, zorder=1)
ax.plot(np.log10(eta), np.log10(rho_eu_32_16), label='Emu', marker='o', linewidth=3, markersize=12, zorder=2)
ax.scatter([np.log10(1e-5)], [np.log10(har_rho_eu_32_16)], label='EMU HAR', marker='x', s=200, c='black', zorder=3)
ax.scatter([np.log10(1e-5)], [np.log10(hsr_rho_eu_64_32)], label=r'EMU HSR + $\eta_\mathrm{matter}=1e-2$', marker='x', s=200, c='red', zorder=3)
ax.axhline(0, color='gray', linewidth=3, linestyle='--')
ax.axvline(0, color='gray', linewidth=3, linestyle='--')
ax.set_xlabel(r'$\log \eta$')
ax.set_ylabel(r'$\log \rho_{eu}$')
ax.set_title('Quantum coherence in the disk')
leg = ax.legend(framealpha=0.0, ncol=1, fontsize=20)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.show()
plt.close(fig)

In [ ]:
eta_for_energy_plots = [1e-12,1e-11,1e-10,1e-9,1e-8,1e-7,1e-6,1e-5]
energybinsMeV_0_13 = [1.0, 3.0, 5.2382, 8.0097, 11.442, 15.691, 20.953, 27.468, 35.536, 45.525, 57.895, 73.212, 92.178]

rho_eu_for_all_energies_48_48_att_total_1e_min_05_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [4.94917054593381e-06, 1.6497212761530736e-06, 9.448211969578011e-07, 6.178960171413183e-07, 4.3254337990272146e-07, 3.154140025943849e-07, 2.362029759676081e-07, 1.8017914816848613e-07, 1.3927174581408017e-07, 1.0871302981197559e-07, 8.548511333284635e-08, 6.76004018432889e-08, 5.369134260558445e-08]
rho_eu_for_all_energies_32_16_att_total_1e_min_05_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [2.544019762663205e-08, 8.480065661127664e-09, 4.856667635062959e-09, 3.1761736688577817e-09, 2.223404694031014e-09, 1.621324086438644e-09, 1.2141555653170958e-09, 9.261757031379962e-10, 7.158994622739915e-10, 5.588180033226668e-10, 4.3941973427316914e-10, 3.4748660649605843e-10, 2.7599002006703775e-10]

rho_eu_for_all_energies_48_48_att_total_1e_min_06_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [4.94917054593381e-06, 1.6497212761530736e-06, 9.448211969578011e-07, 6.178960171413183e-07, 4.3254337990272146e-07, 3.154140025943849e-07, 2.362029759676081e-07, 1.8017914816848613e-07, 1.3927174581408017e-07, 1.0871302981197559e-07, 8.548511333284635e-08, 6.76004018432889e-08, 5.369134260558445e-08]
rho_eu_for_all_energies_32_16_att_total_1e_min_06_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [2.544019762663205e-08, 8.480065661127664e-09, 4.856667635062959e-09, 3.1761736688577817e-09, 2.223404694031014e-09, 1.621324086438644e-09, 1.2141555653170958e-09, 9.261757031379962e-10, 7.158994622739915e-10, 5.588180033226668e-10, 4.3941973427316914e-10, 3.4748660649605843e-10, 2.7599002006703775e-10]

rho_eu_for_all_energies_48_48_att_total_1e_min_07_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [4.94917054593381e-06, 1.6497212761530736e-06, 9.448211969578011e-07, 6.178960171413183e-07, 4.3254337990272146e-07, 3.154140025943849e-07, 2.362029759676081e-07, 1.8017914816848613e-07, 1.3927174581408017e-07, 1.0871302981197559e-07, 8.548511333284635e-08, 6.76004018432889e-08, 5.369134260558445e-08]
rho_eu_for_all_energies_32_16_att_total_1e_min_07_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [2.544019762663205e-08, 8.480065661127664e-09, 4.856667635062959e-09, 3.1761736688577817e-09, 2.223404694031014e-09, 1.621324086438644e-09, 1.2141555653170958e-09, 9.261757031379962e-10, 7.158994622739915e-10, 5.588180033226668e-10, 4.3941973427316914e-10, 3.4748660649605843e-10, 2.7599002006703775e-10]

rho_eu_for_all_energies_48_48_att_total_1e_min_08_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [4.94917054593381e-06, 1.6497212761530736e-06, 9.448211969578011e-07, 6.178960171413183e-07, 4.3254337990272146e-07, 3.154140025943849e-07, 2.362029759676081e-07, 1.8017914816848613e-07, 1.3927174581408017e-07, 1.0871302981197559e-07, 8.548511333284635e-08, 6.76004018432889e-08, 5.369134260558445e-08]
rho_eu_for_all_energies_32_16_att_total_1e_min_08_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [2.544019762663205e-08, 8.480065661127664e-09, 4.856667635062959e-09, 3.1761736688577817e-09, 2.223404694031014e-09, 1.621324086438644e-09, 1.2141555653170958e-09, 9.261757031379962e-10, 7.158994622739915e-10, 5.588180033226668e-10, 4.3941973427316914e-10, 3.4748660649605843e-10, 2.7599002006703775e-10]

rho_eu_for_all_energies_48_48_att_total_1e_min_09_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [4.94917054593381e-06, 1.6497212761530736e-06, 9.448211969578011e-07, 6.178960171413183e-07, 4.3254337990272146e-07, 3.154140025943849e-07, 2.362029759676081e-07, 1.8017914816848613e-07, 1.3927174581408017e-07, 1.0871302981197559e-07, 8.548511333284635e-08, 6.76004018432889e-08, 5.369134260558445e-08]
rho_eu_for_all_energies_32_16_att_total_1e_min_09_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [2.544019762663205e-08, 8.480065661127664e-09, 4.856667635062959e-09, 3.1761736688577817e-09, 2.223404694031014e-09, 1.621324086438644e-09, 1.2141555653170958e-09, 9.261757031379962e-10, 7.158994622739915e-10, 5.588180033226668e-10, 4.3941973427316914e-10, 3.4748660649605843e-10, 2.7599002006703775e-10]

rho_eu_for_all_energies_48_48_att_total_1e_min_10_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [4.94917054593381e-06, 1.6497212761530736e-06, 9.448211969578011e-07, 6.178960171413183e-07, 4.3254337990272146e-07, 3.154140025943849e-07, 2.362029759676081e-07, 1.8017914816848613e-07, 1.3927174581408017e-07, 1.0871302981197559e-07, 8.548511333284635e-08, 6.76004018432889e-08, 5.369134260558445e-08]
rho_eu_for_all_energies_32_16_att_total_1e_min_10_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [2.544019762663205e-08, 8.480065661127664e-09, 4.856667635062959e-09, 3.1761736688577817e-09, 2.223404694031014e-09, 1.621324086438644e-09, 1.2141555653170958e-09, 9.261757031379962e-10, 7.158994622739915e-10, 5.588180033226668e-10, 4.3941973427316914e-10, 3.4748660649605843e-10, 2.7599002006703775e-10]

rho_eu_for_all_energies_48_48_att_total_1e_min_11_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [4.94917054593381e-06, 1.6497212761530736e-06, 9.448211969578011e-07, 6.178960171413183e-07, 4.3254337990272146e-07, 3.154140025943849e-07, 2.362029759676081e-07, 1.8017914816848613e-07, 1.3927174581408017e-07, 1.0871302981197559e-07, 8.548511333284635e-08, 6.76004018432889e-08, 5.369134260558445e-08]
rho_eu_for_all_energies_32_16_att_total_1e_min_11_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [2.544019762663205e-08, 8.480065661127664e-09, 4.856667635062959e-09, 3.1761736688577817e-09, 2.223404694031014e-09, 1.621324086438644e-09, 1.2141555653170958e-09, 9.261757031379962e-10, 7.158994622739915e-10, 5.588180033226668e-10, 4.3941973427316914e-10, 3.4748660649605843e-10, 2.7599002006703775e-10]

rho_eu_for_all_energies_48_48_att_total_1e_min_12_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [4.94917054593381e-06, 1.6497212761530736e-06, 9.448211969578011e-07, 6.178960171413183e-07, 4.3254337990272146e-07, 3.154140025943849e-07, 2.362029759676081e-07, 1.8017914816848613e-07, 1.3927174581408017e-07, 1.0871302981197559e-07, 8.548511333284635e-08, 6.76004018432889e-08, 5.369134260558445e-08]
rho_eu_for_all_energies_32_16_att_total_1e_min_12_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [2.544019762663205e-08, 8.480065661127664e-09, 4.856667635062959e-09, 3.1761736688577817e-09, 2.223404694031014e-09, 1.621324086438644e-09, 1.2141555653170958e-09, 9.261757031379962e-10, 7.158994622739915e-10, 5.588180033226668e-10, 4.3941973427316914e-10, 3.4748660649605843e-10, 2.7599002006703775e-10]

rho_eu_all_att_48_48 = np.array([
    rho_eu_for_all_energies_48_48_att_total_1e_min_12_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    rho_eu_for_all_energies_48_48_att_total_1e_min_11_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    rho_eu_for_all_energies_48_48_att_total_1e_min_10_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    rho_eu_for_all_energies_48_48_att_total_1e_min_09_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    rho_eu_for_all_energies_48_48_att_total_1e_min_08_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    rho_eu_for_all_energies_48_48_att_total_1e_min_07_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    rho_eu_for_all_energies_48_48_att_total_1e_min_06_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    rho_eu_for_all_energies_48_48_att_total_1e_min_05_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
])

fig, ax = plt.subplots(figsize=(10, 6))
for i in range(len(energybinsMeV_0_13)):
    ax.plot(
        np.log10(eta_for_energy_plots),
        np.log10(rho_eu_all_att_48_48[:, i]),
        label=r'$\sin 2\theta_{\mathrm{eff}}$ for $E = %.3f$' % energybinsMeV_0_13[i],
        linewidth=3
    )
ax.plot(np.log10(eta), np.log10(rho_eu_48_48), label='Emu', marker='o', linewidth=3, markersize=12)
# ax.axhline(0, color='gray', linewidth=3, linestyle='--')
# ax.axvline(0, color='gray', linewidth=3, linestyle='--')
ax.set_xlabel(r'$\log \eta$')
ax.set_ylabel(r'$\log \rho_{eu}$')
ax.set_title('Quantum coherence in the poles')
leg = ax.legend(framealpha=0.0, ncol=2, fontsize=12)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.show()
plt.close(fig)

rho_eu_all_att_32_16 = np.array([
    rho_eu_for_all_energies_32_16_att_total_1e_min_12_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    rho_eu_for_all_energies_32_16_att_total_1e_min_11_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    rho_eu_for_all_energies_32_16_att_total_1e_min_10_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    rho_eu_for_all_energies_32_16_att_total_1e_min_09_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    rho_eu_for_all_energies_32_16_att_total_1e_min_08_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    rho_eu_for_all_energies_32_16_att_total_1e_min_07_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    rho_eu_for_all_energies_32_16_att_total_1e_min_06_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    rho_eu_for_all_energies_32_16_att_total_1e_min_05_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
])

fig, ax = plt.subplots(figsize=(10, 6))
for i in range(len(energybinsMeV_0_13)):
    ax.plot(
        np.log10(eta_for_energy_plots),
        np.log10(rho_eu_all_att_32_16[:,i]),
        label=r'$\sin 2\theta_{\mathrm{eff}}$ for $E = %.3f$' % energybinsMeV_0_13[i],
        linewidth=3
    )
ax.plot(np.log10(eta), np.log10(rho_eu_32_16), label='Emu', marker='o', linewidth=3, markersize=12)
# ax.axhline(0, color='gray', linewidth=3, linestyle='--')
# ax.axvline(0, color='gray', linewidth=3, linestyle='--')
ax.set_xlabel(r'$\log \eta$')
ax.set_ylabel(r'$\log \rho_{eu}$')
ax.set_title('Quantum coherence in the disk')
leg = ax.legend(framealpha=0.0, ncol=2, fontsize=12)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.show()
plt.close(fig)

In [ ]:
frequency_for_all_energies_48_48_att_total_1e_min_12_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [0.03295490665682332, 0.032954951384180003, 0.032954960939872505, 0.03295496537170877, 0.03295496788437307, 0.03295496947219481, 0.032954970545990586, 0.03295497130545762, 0.0329549718600043, 0.032954972274262744, 0.032954972589143816, 0.03295497283159163, 0.03295497302014492]
frequency_for_all_energies_32_16_att_total_1e_min_12_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [6.41109225676465, 6.411092301492364, 6.411092311048087, 6.411092315479931, 6.4110923179925985, 6.411092319580422, 6.411092320654219, 6.411092321413685, 6.411092321968233, 6.4110923223824905, 6.411092322697373, 6.41109232293982, 6.411092323128373]

frequency_for_all_energies_48_48_att_total_1e_min_11_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [0.32954906656823324, 0.3295495138418, 0.32954960939872496, 0.3295496537170877, 0.3295496788437306, 0.32954969472194806, 0.3295497054599059, 0.32954971305457625, 0.329549718600043, 0.32954972274262745, 0.32954972589143816, 0.32954972831591633, 0.3295497302014492]
frequency_for_all_energies_32_16_att_total_1e_min_11_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [64.11092256764651, 64.11092301492366, 64.11092311048085, 64.11092315479931, 64.11092317992599, 64.11092319580422, 64.11092320654217, 64.11092321413685, 64.11092321968232, 64.11092322382491, 64.11092322697372, 64.1109232293982, 64.11092323128375]

frequency_for_all_energies_48_48_att_total_1e_min_10_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [3.295490665682332, 3.295495138417999, 3.2954960939872504, 3.2954965371708775, 3.2954967884373065, 3.295496947219481, 3.2954970545990596, 3.2954971305457623, 3.2954971860004303, 3.2954972274262744, 3.295497258914381, 3.295497283159164, 3.295497302014492]
frequency_for_all_energies_32_16_att_total_1e_min_10_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [641.1092256764654, 641.1092301492365, 641.1092311048087, 641.1092315479931, 641.1092317992599, 641.1092319580423, 641.1092320654219, 641.1092321413686, 641.1092321968232, 641.1092322382491, 641.1092322697374, 641.109232293982, 641.1092323128373]

frequency_for_all_energies_48_48_att_total_1e_min_09_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [32.95490665682332, 32.95495138418, 32.95496093987251, 32.95496537170877, 32.95496788437307, 32.95496947219481, 32.95497054599059, 32.95497130545762, 32.9549718600043, 32.954972274262744, 32.95497258914382, 32.95497283159164, 32.95497302014492]
frequency_for_all_energies_32_16_att_total_1e_min_09_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [6411.092256764651, 6411.092301492365, 6411.092311048087, 6411.092315479931, 6411.092317992599, 6411.092319580423, 6411.092320654219, 6411.092321413686, 6411.092321968233, 6411.092322382492, 6411.092322697373, 6411.09232293982, 6411.092323128373]

frequency_for_all_energies_48_48_att_total_1e_min_08_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [329.5490665682331, 329.54951384180004, 329.54960939872507, 329.5496537170877, 329.54967884373065, 329.5496947219481, 329.5497054599059, 329.54971305457616, 329.54971860004304, 329.5497227426275, 329.5497258914382, 329.5497283159163, 329.54973020144917]
frequency_for_all_energies_32_16_att_total_1e_min_08_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [64110.9225676465, 64110.923014923654, 64110.92311048087, 64110.92315479931, 64110.92317992598, 64110.92319580422, 64110.92320654219, 64110.92321413686, 64110.92321968233, 64110.92322382491, 64110.92322697374, 64110.9232293982, 64110.923231283734]

frequency_for_all_energies_48_48_att_total_1e_min_07_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [3295.4906656823314, 3295.495138418, 3295.4960939872503, 3295.496537170877, 3295.496788437306, 3295.49694721948, 3295.4970545990586, 3295.497130545762, 3295.4971860004307, 3295.4972274262736, 3295.4972589143813, 3295.4972831591635, 3295.4973020144917]
frequency_for_all_energies_32_16_att_total_1e_min_07_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [641109.225676465, 641109.2301492364, 641109.2311048087, 641109.2315479931, 641109.2317992598, 641109.2319580422, 641109.232065422, 641109.2321413686, 641109.2321968232, 641109.2322382492, 641109.2322697373, 641109.232293982, 641109.2323128373]

frequency_for_all_energies_48_48_att_total_1e_min_06_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [32954.90665682332, 32954.95138417999, 32954.9609398725, 32954.96537170877, 32954.967884373065, 32954.96947219481, 32954.970545990596, 32954.97130545761, 32954.9718600043, 32954.972274262735, 32954.972589143814, 32954.972831591636, 32954.97302014492]
frequency_for_all_energies_32_16_att_total_1e_min_06_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [6411092.25676465, 6411092.301492364, 6411092.311048087, 6411092.315479931, 6411092.317992599, 6411092.319580421, 6411092.320654219, 6411092.321413686, 6411092.321968232, 6411092.322382492, 6411092.322697373, 6411092.322939821, 6411092.323128372]

frequency_for_all_energies_48_48_att_total_1e_min_05_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [329549.0665682332, 329549.51384180004, 329549.6093987251, 329549.65371708776, 329549.67884373065, 329549.69472194806, 329549.7054599059, 329549.71305457625, 329549.7186000431, 329549.7227426275, 329549.7258914382, 329549.7283159163, 329549.7302014491]
frequency_for_all_energies_32_16_att_total_1e_min_05_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [64110922.56764651, 64110923.014923654, 64110923.11048088, 64110923.15479932, 64110923.17992599, 64110923.19580423, 64110923.206542194, 64110923.214136854, 64110923.219682336, 64110923.22382493, 64110923.22697374, 64110923.22939821, 64110923.23128374]

frequency_all_att_48_48 = np.array([
    frequency_for_all_energies_48_48_att_total_1e_min_12_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_for_all_energies_48_48_att_total_1e_min_11_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_for_all_energies_48_48_att_total_1e_min_10_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_for_all_energies_48_48_att_total_1e_min_09_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_for_all_energies_48_48_att_total_1e_min_08_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_for_all_energies_48_48_att_total_1e_min_07_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_for_all_energies_48_48_att_total_1e_min_06_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_for_all_energies_48_48_att_total_1e_min_05_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
])

fig, ax = plt.subplots(figsize=(10, 6))
for i in range(len(energybinsMeV_0_13)):
    ax.plot(np.log10(eta_for_energy_plots), np.log10(frequency_all_att_48_48[:,i]/pc.PhysConst.c), label=f'$E = {energybinsMeV_0_13[i]}$', linewidth=3)
# ax.axhline(0, color='gray', linewidth=3, linestyle='--')
# ax.axvline(0, color='gray', linewidth=3, linestyle='--')
ax.set_xlabel(r'$\log \eta$')
ax.set_ylabel(r'$\log\,\omega\,[\,\mathrm{cm}^{-1}\,]$')
ax.set_title('Effective frequency in the poles')
leg = ax.legend(framealpha=0.0, ncol=1, fontsize=12)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.show()
plt.close(fig)

frequency_all_att_32_16 = np.array([
    frequency_for_all_energies_32_16_att_total_1e_min_12_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_for_all_energies_32_16_att_total_1e_min_11_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_for_all_energies_32_16_att_total_1e_min_10_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_for_all_energies_32_16_att_total_1e_min_09_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_for_all_energies_32_16_att_total_1e_min_08_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_for_all_energies_32_16_att_total_1e_min_07_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_for_all_energies_32_16_att_total_1e_min_06_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_for_all_energies_32_16_att_total_1e_min_05_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
])

fig, ax = plt.subplots(figsize=(10, 6))
for i in range(len(energybinsMeV_0_13)):
    ax.plot(np.log10(eta_for_energy_plots), np.log10(frequency_all_att_32_16[:,i]/pc.PhysConst.c), label=f'$E = {energybinsMeV_0_13[i]}$', linewidth=3)
# ax.axhline(0, color='gray', linewidth=3, linestyle='--')
# ax.axvline(0, color='gray', linewidth=3, linestyle='--')
ax.set_xlabel(r'$\log \eta$')
ax.set_ylabel(r'$\log\,\omega\,[\,\mathrm{cm}^{-1}\,]$')
ax.set_title('Effective frequency in the disk')
leg = ax.legend(framealpha=0.0, ncol=1, fontsize=12)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.show()
plt.close(fig)